### Building a RAG System with LangChain and FAISS 
Introduction to RAG (Retrieval-Augmented Generation)
RAG combines the power of retrieval systems with generative AI models. Instead of relying solely on the model's training data, RAG:

1. Retrieves relevant documents from a knowledge base
2. Uses these documents as context for the LLM
3. Generates responses based on both the retrieved context and the model's knowledge

### FAISS 
https://github.com/facebookresearch/faiss

FAISS is a library for efficient similarity search and clustering of dense vectors.

Key advantages:
1. Extremely fast similarity search
2. Memory efficient
3. Supports GPU acceleration
4. Can handle millions of vectors

How it works:
- Indexes vectors for fast nearest neighbor search
- Returns most similar vectors based on distance metrics


In [1]:
import os
from dotenv import load_dotenv
import numpy as np
from typing import List, Dict, Optional
load_dotenv()


True

In [2]:
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate,PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain.messages import AIMessage,HumanMessage


c:\Users\Shounak\OneDrive\Desktop\School\AI\Project_1\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_groq import ChatGroq
from langchain_community.vectorstores import FAISS
from langchain_community.document_loaders import Docx2txtLoader,TextLoader,DirectoryLoader


### Data Ingestion And Processing


In [4]:
## Method1: Using Docx2txtLoader
try:
    docx_loader = DirectoryLoader(
        path = "data",
        glob="*.docx",
        loader_cls=Docx2txtLoader
    )
    docs = docx_loader.load()
    print(f"✅ Loaded {len(docs)} document(s)")
    print(f"Content preview: {docs[0].page_content[:200]}...")
    print(f"Metadata: {docs[0].metadata}")

except Exception as e:
    print(f"Error: {e}")

✅ Loaded 6 document(s)
Content preview: Name : Shounak S Chavan. 

Roll no: 321045 

Div : TY-A

Batch : A2 

PRN no.: 22311260 

Assignment 1 (Part-2)-(Remaining commands): 

Practise all Linux commands for Cloud and DevOps

Aim

The aim o...
Metadata: {'source': 'data\\A2_321045_Assign1(Part-2)_CC_Shounak_Chavan.docx'}


In [5]:
docs


[Document(metadata={'source': 'data\\A2_321045_Assign1(Part-2)_CC_Shounak_Chavan.docx'}, page_content='Name : Shounak S Chavan. \n\nRoll no: 321045 \n\nDiv : TY-A\n\nBatch : A2 \n\nPRN no.: 22311260 \n\nAssignment 1 (Part-2)-(Remaining commands): \n\nPractise all Linux commands for Cloud and DevOps\n\nAim\n\nThe aim of this assignment is to study and understand the basic Linux commands used in Cloud and DevOps environments. This helps students and professionals to manage cloud-based infrastructure, monitor system performance, and deploy and maintain applications efficiently using the Linux operating system.\n\n\n\nTheory\n\nLinux commands are a set of instructions used to perform specific tasks on a Linux operating system. These commands are executed through the Command Line Interface (CLI), which is a text-based interface that allows users to interact directly with the system.\n\nLinux commands follow a standard syntax consisting of a command name, options, and arguments.\nThe command

### Document Splitting

In [6]:
# Initialize text splitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=600,
    chunk_overlap=50,
    length_function=len,
    separators=["\n\n", "\n", " ", ""]
)
chunks = text_splitter.split_documents(docs)
print(f"Split into {len(chunks)} chunks")

Split into 37 chunks


In [7]:
chunks

[Document(metadata={'source': 'data\\A2_321045_Assign1(Part-2)_CC_Shounak_Chavan.docx'}, page_content='Name : Shounak S Chavan. \n\nRoll no: 321045 \n\nDiv : TY-A\n\nBatch : A2 \n\nPRN no.: 22311260 \n\nAssignment 1 (Part-2)-(Remaining commands): \n\nPractise all Linux commands for Cloud and DevOps\n\nAim\n\nThe aim of this assignment is to study and understand the basic Linux commands used in Cloud and DevOps environments. This helps students and professionals to manage cloud-based infrastructure, monitor system performance, and deploy and maintain applications efficiently using the Linux operating system.\n\n\n\nTheory'),
 Document(metadata={'source': 'data\\A2_321045_Assign1(Part-2)_CC_Shounak_Chavan.docx'}, page_content='Theory\n\nLinux commands are a set of instructions used to perform specific tasks on a Linux operating system. These commands are executed through the Command Line Interface (CLI), which is a text-based interface that allows users to interact directly with the syst

In [8]:
chunks[0]

Document(metadata={'source': 'data\\A2_321045_Assign1(Part-2)_CC_Shounak_Chavan.docx'}, page_content='Name : Shounak S Chavan. \n\nRoll no: 321045 \n\nDiv : TY-A\n\nBatch : A2 \n\nPRN no.: 22311260 \n\nAssignment 1 (Part-2)-(Remaining commands): \n\nPractise all Linux commands for Cloud and DevOps\n\nAim\n\nThe aim of this assignment is to study and understand the basic Linux commands used in Cloud and DevOps environments. This helps students and professionals to manage cloud-based infrastructure, monitor system performance, and deploy and maintain applications efficiently using the Linux operating system.\n\n\n\nTheory')

### Embedding Models

In [9]:
embedding= HuggingFaceEmbeddings(
    model = "sentence-transformers/all-MiniLM-L6-v2",

)
embedding

HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2', cache_folder=None, model_kwargs={}, encode_kwargs={}, query_encode_kwargs={}, multi_process=False, show_progress=False)

In [10]:
sample_text="Hi This is a sample text to test the embedding generation."
embeddings = embedding.embed_query(sample_text)

In [11]:
embeddings

[-0.043723214417696,
 -0.019568169489502907,
 -0.0007140834350138903,
 0.024498464539647102,
 0.03770345821976662,
 0.04957782104611397,
 -0.06052356958389282,
 -0.042971499264240265,
 0.0032796126324683428,
 -0.03722921758890152,
 0.050972942262887955,
 0.0025392223615199327,
 0.06324561685323715,
 -0.03421986848115921,
 -0.06161472201347351,
 0.058626413345336914,
 0.08862781524658203,
 -0.010152316652238369,
 -0.019198153167963028,
 -0.022553468123078346,
 0.042485833168029785,
 0.08119737356901169,
 0.07353749871253967,
 -0.06787510216236115,
 0.049007706344127655,
 -0.02178381010890007,
 -0.05650569498538971,
 0.06915799528360367,
 0.11375511437654495,
 -0.01839854195713997,
 0.09425162523984909,
 0.0254929531365633,
 0.04581533744931221,
 0.07101503759622574,
 0.0452791191637516,
 0.10016463696956635,
 -0.021224714815616608,
 0.04691961035132408,
 0.024678871035575867,
 0.06505370885133743,
 -0.01632518135011196,
 -0.015586888417601585,
 0.03707626089453697,
 0.06086057797074318,

### Intilialize the FAISS Vector Store And Stores the chunks in Vector Representation

In [12]:

vectorstore = FAISS.from_documents(
    documents=chunks,
    embedding=embedding,

)
print(f"vectorstore created with {vectorstore.index.ntotal} vectors")

vectorstore created with 37 vectors


In [13]:
# Save vectorstore to disk
vectorstore.save_local("faiss_index")

In [14]:
loaded_vectorstore = FAISS.load_local(
    "faiss_index", 
    embedding,
    allow_dangerous_deserialization=True
    )
print(f"Loaded vectorstore with {loaded_vectorstore.index.ntotal} vectors")

Loaded vectorstore with 37 vectors


In [15]:
## Similarity search
query = "What is my name?"

results = loaded_vectorstore.similarity_search(query,k=3)
print(results)

[Document(id='e70d0367-5d86-4bc7-920d-74b633b51dfb', metadata={'source': 'data\\A2_321045_Assign2_CC_Shounak_Chavan.docx'}, page_content='Name : Shounak S Chavan. \n\nRoll no: 321045\n\nDiv : TY-A\n\nBatch : A2\n\nPRN no.: 22311260\n\n\n\nAssignment 2:\n\n\n\nAIM :\n\n\n\nThe aim of this assignment is to understand the fundamentals of shell scripting and to develop the ability to write simple shell programs for automating common tasks in a Linux operating system. This assignment helps students learn how to interact with the system using commands, create scripts, and improve efficiency in system administration.\n\n\n\nTHEORY :'), Document(id='4e2e2a24-7c9d-4307-b677-7fd369c75982', metadata={'source': 'data\\A2_321045_Assign3_CC_Shounak_Chavan.docx'}, page_content='Name : Shounak S Chavan. Roll no: 321045\n\nDiv : TY-A\n\nBatch : A2\n\nPRN no.: 22311260\n\n\n\nAssignment 3:\n\n\n\nAIM :\n\n\n\nTo design and deploy a highly available web application on the AWS cloud platform by utilizing 

In [16]:
print(f"Query: {query}\n")
print("Top 3 similar chunks:")
for i, doc in enumerate(results):
    print(f"\n{i+1}. Source: {doc.metadata['source']}")
    print(f"   Content: {doc.page_content[:200]}...")

Query: What is my name?

Top 3 similar chunks:

1. Source: data\A2_321045_Assign2_CC_Shounak_Chavan.docx
   Content: Name : Shounak S Chavan. 

Roll no: 321045

Div : TY-A

Batch : A2

PRN no.: 22311260



Assignment 2:



AIM :



The aim of this assignment is to understand the fundamentals of shell scripting and t...

2. Source: data\A2_321045_Assign3_CC_Shounak_Chavan.docx
   Content: Name : Shounak S Chavan. Roll no: 321045

Div : TY-A

Batch : A2

PRN no.: 22311260



Assignment 3:



AIM :



To design and deploy a highly available web application on the AWS cloud platform by ut...

3. Source: data\A2_321045_Assign1(Part-2)_CC_Shounak_Chavan.docx
   Content: 9.hostname: 

The hostname command shows the name assigned to the system.
It identifies the device on a network.



10.netstat: 

The netstat command displays active network connections, listening por...


In [17]:
query = "Shounak Chavan"
results = loaded_vectorstore.similarity_search(query, k=3)
print(results)

[Document(id='e70d0367-5d86-4bc7-920d-74b633b51dfb', metadata={'source': 'data\\A2_321045_Assign2_CC_Shounak_Chavan.docx'}, page_content='Name : Shounak S Chavan. \n\nRoll no: 321045\n\nDiv : TY-A\n\nBatch : A2\n\nPRN no.: 22311260\n\n\n\nAssignment 2:\n\n\n\nAIM :\n\n\n\nThe aim of this assignment is to understand the fundamentals of shell scripting and to develop the ability to write simple shell programs for automating common tasks in a Linux operating system. This assignment helps students learn how to interact with the system using commands, create scripts, and improve efficiency in system administration.\n\n\n\nTHEORY :'), Document(id='3ded781b-9a76-468a-89aa-78de61409673', metadata={'source': 'data\\A2_321045_Assign7_CC_Shounak_Chavan.docx'}, page_content='Name : Shounak S Chavan. Roll no: 321045\n\nDiv : TY-A\n\nBatch : A2\n\nPRN no.: 22311260\n\n\n\nAssignment 7:\n\n\n\nAIM :\n\n\n\nTo develop and deploy a Python FastAPI web application using Docker, create a containerized ima

In [18]:
print(f"Query: {query}\n")
print("Top 3 similar chunks:")
for i, doc in enumerate(results):
    print(f"\n{i+1}. Source: {doc.metadata['source']}")
    print(f"   Content: {doc.page_content[:200]}...")

Query: Shounak Chavan

Top 3 similar chunks:

1. Source: data\A2_321045_Assign2_CC_Shounak_Chavan.docx
   Content: Name : Shounak S Chavan. 

Roll no: 321045

Div : TY-A

Batch : A2

PRN no.: 22311260



Assignment 2:



AIM :



The aim of this assignment is to understand the fundamentals of shell scripting and t...

2. Source: data\A2_321045_Assign7_CC_Shounak_Chavan.docx
   Content: Name : Shounak S Chavan. Roll no: 321045

Div : TY-A

Batch : A2

PRN no.: 22311260



Assignment 7:



AIM :



To develop and deploy a Python FastAPI web application using Docker, create a container...

3. Source: data\A2_321045_Assign3_CC_Shounak_Chavan.docx
   Content: Name : Shounak S Chavan. Roll no: 321045

Div : TY-A

Batch : A2

PRN no.: 22311260



Assignment 3:



AIM :



To design and deploy a highly available web application on the AWS cloud platform by ut...


### Advanced Similarity Search With Scores

In [19]:
results_scores=loaded_vectorstore.similarity_search_with_score(query,k=3)
results_scores

[(Document(id='e70d0367-5d86-4bc7-920d-74b633b51dfb', metadata={'source': 'data\\A2_321045_Assign2_CC_Shounak_Chavan.docx'}, page_content='Name : Shounak S Chavan. \n\nRoll no: 321045\n\nDiv : TY-A\n\nBatch : A2\n\nPRN no.: 22311260\n\n\n\nAssignment 2:\n\n\n\nAIM :\n\n\n\nThe aim of this assignment is to understand the fundamentals of shell scripting and to develop the ability to write simple shell programs for automating common tasks in a Linux operating system. This assignment helps students learn how to interact with the system using commands, create scripts, and improve efficiency in system administration.\n\n\n\nTHEORY :'),
  np.float32(1.3268948)),
 (Document(id='3ded781b-9a76-468a-89aa-78de61409673', metadata={'source': 'data\\A2_321045_Assign7_CC_Shounak_Chavan.docx'}, page_content='Name : Shounak S Chavan. Roll no: 321045\n\nDiv : TY-A\n\nBatch : A2\n\nPRN no.: 22311260\n\n\n\nAssignment 7:\n\n\n\nAIM :\n\n\n\nTo develop and deploy a Python FastAPI web application using Docke

In [20]:
loaded_vectorstore.similarity_search_with_score("Assignment 3", k=10)

[(Document(id='e70d0367-5d86-4bc7-920d-74b633b51dfb', metadata={'source': 'data\\A2_321045_Assign2_CC_Shounak_Chavan.docx'}, page_content='Name : Shounak S Chavan. \n\nRoll no: 321045\n\nDiv : TY-A\n\nBatch : A2\n\nPRN no.: 22311260\n\n\n\nAssignment 2:\n\n\n\nAIM :\n\n\n\nThe aim of this assignment is to understand the fundamentals of shell scripting and to develop the ability to write simple shell programs for automating common tasks in a Linux operating system. This assignment helps students learn how to interact with the system using commands, create scripts, and improve efficiency in system administration.\n\n\n\nTHEORY :'),
  np.float32(1.250057)),
 (Document(id='d951797f-3f64-49f7-9349-06f69e37017f', metadata={'source': 'data\\A2_321045_Assign2_CC_Shounak_Chavan.docx'}, page_content='Output :\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\nCONCLUSION:\n\n\n\nThis assignment provides a clear understanding of shell scripting and its importance in the Linux operating system. By writing and executin

#### Initialize LLM, RAG Chain, Prompt Template,Query the RAG system

In [21]:
model = ChatGroq(
    model = 'llama-3.1-8b-instant',
    temperature=0.3,
    max_tokens=200
)
msg = model.invoke("hi")
msg.content

'How can I assist you today?'

### Create RAG Chain  - Using LCEL (LangChain Expression Language)

In [22]:
# 1. Simple RAG Chain with LCEL
simple_prompt = ChatPromptTemplate.from_template("""Answer the question based only on the following context:
Context: {context}

Question: {question}

Answer:""")

In [23]:
retriever = loaded_vectorstore.as_retriever(
    search_type = "similarity",
    search_kwargs={'k':5}
)

In [24]:
retriever

VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x0000018FD32C56D0>, search_kwargs={'k': 5})

In [25]:
## Format the output documents for the prompt
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

In [26]:
simple_rag_chain = (
    {"context":retriever | format_docs,"question":RunnablePassthrough()}
    | simple_prompt
    | model
    |StrOutputParser()
)

In [27]:
simple_rag_chain

{
  context: VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x0000018FD32C56D0>, search_kwargs={'k': 5})
           | RunnableLambda(format_docs),
  question: RunnablePassthrough()
}
| ChatPromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template='Answer the question based only on the following context:\nContext: {context}\n\nQuestion: {question}\n\nAnswer:'), additional_kwargs={})])
| ChatGroq(profile={'max_input_tokens': 131072, 'max_output_tokens': 8192, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x0000018FD69B6A50>

In [28]:
### Conversational Rag Chain

conversational_prompt = ChatPromptTemplate.from_messages([
    ("system","You are a helpful AI assistant. Use the provided context to answer questions."),
    ("placeholder","{chat_history}"),
    ("human","Context:{context}\n\nQuestion:{input}")
])


In [29]:
def create_conversational_rag():
    return (
        RunnablePassthrough.assign(
            context = lambda x: format_docs(retriever.invoke(x["input"]))
        )
        | conversational_prompt
        | model
        | StrOutputParser()
    )

conversational_rag = create_conversational_rag()

In [30]:
conversational_rag

RunnableAssign(mapper={
  context: RunnableLambda(lambda x: format_docs(retriever.invoke(x['input'])))
})
| ChatPromptTemplate(input_variables=['context', 'input'], optional_variables=['chat_history'], input_types={'chat_history': list[typing.Annotated[typing.Union[typing.Annotated[langchain_core.messages.ai.AIMessage, Tag(tag='ai')], typing.Annotated[langchain_core.messages.human.HumanMessage, Tag(tag='human')], typing.Annotated[langchain_core.messages.chat.ChatMessage, Tag(tag='chat')], typing.Annotated[langchain_core.messages.system.SystemMessage, Tag(tag='system')], typing.Annotated[langchain_core.messages.function.FunctionMessage, Tag(tag='function')], typing.Annotated[langchain_core.messages.tool.ToolMessage, Tag(tag='tool')], typing.Annotated[langchain_core.messages.ai.AIMessageChunk, Tag(tag='AIMessageChunk')], typing.Annotated[langchain_core.messages.human.HumanMessageChunk, Tag(tag='HumanMessageChunk')], typing.Annotated[langchain_core.messages.chat.ChatMessageChunk, Tag(tag=

In [31]:
### streaming RAG chain
streaming_rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | simple_prompt
    | model
)

print("Modern RAG chains created successfully!")
print("Available chains:")
print("- simple_rag_chain: Basic Q&A")
print("- conversational_rag: Maintains conversation history")
print("- streaming_rag_chain: Supports token streaming")

Modern RAG chains created successfully!
Available chains:
- simple_rag_chain: Basic Q&A
- conversational_rag: Maintains conversation history
- streaming_rag_chain: Supports token streaming


In [32]:
# Test function for different chain types
def test_rag_chains(question: str):
    """Test all RAG chain variants"""
    print(f"Question: {question}")
    print("=" * 80)
    
    # 1. Simple RAG
    print("\n1. Simple RAG Chain:")
    answer = simple_rag_chain.invoke(question)
    print(f"Answer: {answer}")

    print("\n2. Streaming RAG:")
    print("Answer: ", end="", flush=True)
    for chunk in streaming_rag_chain.stream(question):
        print(chunk.content, end="", flush=True)
    print()

In [33]:
test_rag_chains("tell me about assignment 5")

Question: tell me about assignment 5

1. Simple RAG Chain:
Answer: There is no information about Assignment 5 in the provided context. The context only mentions Assignments 2 and 1 (Part-2), but not Assignment 5.

2. Streaming RAG:
Answer: There is no mention of Assignment 5 in the provided context. The context only mentions Assignment 1 (Part-2) and Assignment 2, but there is no information about Assignment 5.


In [34]:
# Test with multiple questions
test_questions = [
    "Difference between assignment 1 and assignment 2",
    "explain about grep command in which assignmet it is there "
]

for question in test_questions:
    print("\n" + "=" * 80 + "\n")
    test_rag_chains(question)



Question: Difference between assignment 1 and assignment 2

1. Simple RAG Chain:
Answer: Based on the provided context, it appears that Assignment 1 and Assignment 2 are two separate tasks given to Shounak S Chavan.

Assignment 1 seems to be focused on practicing Linux commands for Cloud and DevOps, while Assignment 2 is focused on understanding the fundamentals of shell scripting and developing the ability to write simple shell programs for automating common tasks in a Linux operating system.

Therefore, the main difference between Assignment 1 and Assignment 2 is their focus:

- Assignment 1: Linux commands for Cloud and DevOps
- Assignment 2: Shell scripting and automating system tasks

2. Streaming RAG:
Answer: Based on the given context, it appears that Assignment 1 and Assignment 2 are two separate assignments with different aims.

Assignment 1 aims to study and understand basic Linux commands used in Cloud and DevOps environments, while Assignment 2 aims to understand the fund

In [35]:
## Conversational example
print("\n3. Conversational RAG Example:")
chat_history = []

# First question
q1 = "What is assignment 3?"
a1 = conversational_rag.invoke({
    "input": q1,
    "chat_history": chat_history
})

print(f"Q1: {q1}")
print(f"A1: {a1}")


3. Conversational RAG Example:


Q1: What is assignment 3?
A1: Unfortunately, the provided context does not mention Assignment 3. It only mentions Assignment 1 (Part-2) and Assignment 2.


In [36]:
# Update history
chat_history.extend([
    HumanMessage(content=q1),
    AIMessage(content=a1)
])

In [37]:
# Follow-up question
q2 = "which assignment has ec2 instance in it"
a2 = conversational_rag.invoke({
    "input": q2,
    "chat_history": chat_history
})
print(f"\nQ2: {q2}")
print(f"A2: {a2}")


Q2: which assignment has ec2 instance in it
A2: Based on the provided context, the assignment that has an EC2 instance in it is Assignment 3.


In [38]:
# Update history
chat_history.extend([
    HumanMessage(content=q2),
    AIMessage(content=a2)
])